In [6]:
# INSTALL
!pip install duckdb pandas prettytable

In [29]:
#IMPORTS
import pandas as pd
import duckdb

In [30]:
#LOAD DATA
df = pd.read_csv("ER Wait Time Dataset.csv")
print(f"Dataset shape:{df.shape}")
df.head()

Dataset shape:(5000, 19)


,Visit ID,Patient ID,Hospital ID,Hospital Name,Region,Visit Date,Day of Week,Season,Time of Day,Urgency Level,Nurse-to-Patient Ratio,Specialist Availability,Facility Size (Beds),Time to Registration (min),Time to Triage (min),Time to Medical Professional (min),Total Wait Time (min),Patient Outcome,Patient Satisfaction
0,HOSP-1-20240210-0001,PAT-00001,HOSP-1,Springfield General Hospital,Urban,10-02-2024 20:20,Saturday,Winter,Late Morning,Medium,4,3,92,17,22,66,105,Discharged,1
1,HOSP-3-20241128-0001,PAT-00002,HOSP-3,Northside Community Hospital,Rural,28-11-2024 02:07,Thursday,Fall,Evening,Medium,4,0,38,9,30,30,69,Discharged,3
2,HOSP-3-20240930-0002,PAT-00003,HOSP-3,Northside Community Hospital,Rural,30-09-2024 04:02,Monday,Fall,Evening,Low,5,1,38,38,40,125,203,Discharged,1
3,HOSP-2-20240227-0001,PAT-00004,HOSP-2,Riverside Medical Center,Urban,27-02-2024 00:31,Tuesday,Winter,Evening,High,4,5,94,8,16,64,88,Discharged,2
4,HOSP-1-20240306-0002,PAT-00005,HOSP-1,Springfield General Hospital,Urban,06-03-2024 16:52,Wednesday,Spring,Afternoon,Low,4,8,74,26,29,63,118,Discharged,1


In [31]:
# PREPROCESSING
df['Visit Date'] = pd.to_datetime(df['Visit Date'], dayfirst=True, errors='coerce')
print(df.dtypes)

Visit ID                                      object
Patient ID                                    object
Hospital ID                                   object
Hospital Name                                 object
Region                                        object
Visit Date                            datetime64[ns]
Day of Week                                   object
Season                                        object
Time of Day                                   object
Urgency Level                                 object
Nurse-to-Patient Ratio                         int64
Specialist Availability                        int64
Facility Size (Beds)                           int64
Time to Registration (min)                     int64
Time to Triage (min)                           int64
Time to Medical Professional (min)             int64
Total Wait Time (min)                          int64
Patient Outcome                               object
Patient Satisfaction                          

In [32]:
#CREATE DIMENSION TABLES
dim_patient = df[['Patient ID']].drop_duplicates().reset_index(drop=True)
print(f"dim_patient:{dim_patient.shape}")
dim_patient.head()

dim_patient:(5000, 1)


,Patient ID
0,PAT-00001
1,PAT-00002
2,PAT-00003
3,PAT-00004
4,PAT-00005


In [33]:
dim_hospital = df[['Hospital ID', 'Hospital Name', 'Region']].drop_duplicates(subset='Hospital ID').reset_index(drop=True)
print(f"dim_hospital:{dim_hospital.shape}")
dim_hospital.head()

dim_hospital:(5, 3)


,Hospital ID,Hospital Name,Region
0,HOSP-1,Springfield General Hospital,Urban
1,HOSP-3,Northside Community Hospital,Rural
2,HOSP-2,Riverside Medical Center,Urban
3,HOSP-5,Summit Health Center,Urban
4,HOSP-4,St. Mary’s Regional Health,Rural


In [34]:
dim_time = df[['Visit Date', 'Day of Week', 'Season', 'Time of Day']].drop_duplicates().reset_index(drop=True)
print(f"dim_time:{dim_time.shape}")
dim_time.head()

dim_time:(4995, 4)


,Visit Date,Day of Week,Season,Time of Day
0,2024-02-10 20:20:00,Saturday,Winter,Late Morning
1,2024-11-28 02:07:00,Thursday,Fall,Evening
2,2024-09-30 04:02:00,Monday,Fall,Evening
3,2024-02-27 00:31:00,Tuesday,Winter,Evening
4,2024-03-06 16:52:00,Wednesday,Spring,Afternoon


In [35]:
#CREATE FACT TABLE
fact_er_visits = df[[
    'Visit ID', 'Patient ID', 'Hospital ID', 'Visit Date',
    'Nurse-to-Patient Ratio', 'Specialist Availability', 'Facility Size (Beds)',
    'Time to Registration (min)', 'Time to Triage (min)',
    'Time to Medical Professional (min)', 'Total Wait Time (min)',
    'Patient Outcome', 'Patient Satisfaction', 'Urgency Level'
]]
print(f"fact_er_visits: {fact_er_visits.shape}")
fact_er_visits.head()

fact_er_visits: (5000, 14)


,Visit ID,Patient ID,Hospital ID,Visit Date,Nurse-to-Patient Ratio,Specialist Availability,Facility Size (Beds),Time to Registration (min),Time to Triage (min),Time to Medical Professional (min),Total Wait Time (min),Patient Outcome,Patient Satisfaction,Urgency Level
0,HOSP-1-20240210-0001,PAT-00001,HOSP-1,2024-02-10 20:20:00,4,3,92,17,22,66,105,Discharged,1,Medium
1,HOSP-3-20241128-0001,PAT-00002,HOSP-3,2024-11-28 02:07:00,4,0,38,9,30,30,69,Discharged,3,Medium
2,HOSP-3-20240930-0002,PAT-00003,HOSP-3,2024-09-30 04:02:00,5,1,38,38,40,125,203,Discharged,1,Low
3,HOSP-2-20240227-0001,PAT-00004,HOSP-2,2024-02-27 00:31:00,4,5,94,8,16,64,88,Discharged,2,High
4,HOSP-1-20240306-0002,PAT-00005,HOSP-1,2024-03-06 16:52:00,4,8,74,26,29,63,118,Discharged,1,Low


In [36]:
# LOAD INTO DUCKDB
con = duckdb.connect('ER_Visits.duckdb')

con.execute("CREATE OR REPLACE TABLE ER_Wait_Time_original AS SELECT * FROM df")
con.execute("CREATE OR REPLACE TABLE dim_patient AS SELECT * FROM dim_patient")
con.execute("CREATE OR REPLACE TABLE dim_hospital AS SELECT * FROM dim_hospital")
con.execute("CREATE OR REPLACE TABLE dim_time AS SELECT * FROM dim_time")
con.execute("CREATE OR REPLACE TABLE fact_er_visits AS SELECT * FROM fact_er_visits")

print("All tables loaded successfully")

All tables loaded successfully


In [38]:
# VERIFY
con.execute("""
    SELECT 'fact_er_visits' AS table_name, COUNT(*) AS total_rows
    FROM fact_er_visits
    UNION ALL
    SELECT 'dim_patient', COUNT(*) FROM dim_patient
    UNION ALL
    SELECT 'dim_hospital', COUNT(*) FROM dim_hospital
    UNION ALL
    SELECT 'dim_time', COUNT(*) FROM dim_time
""").df()

,table_name,total_rows
0,fact_er_visits,5000
1,dim_patient,5000
2,dim_hospital,5
3,dim_time,4995


In [39]:
#PREVIEW ORIGINAL DATA
con.execute("SELECT * FROM ER_Wait_Time_original LIMIT 5").df()

,Visit ID,Patient ID,Hospital ID,Hospital Name,Region,Visit Date,Day of Week,Season,Time of Day,Urgency Level,Nurse-to-Patient Ratio,Specialist Availability,Facility Size (Beds),Time to Registration (min),Time to Triage (min),Time to Medical Professional (min),Total Wait Time (min),Patient Outcome,Patient Satisfaction
0,HOSP-1-20240210-0001,PAT-00001,HOSP-1,Springfield General Hospital,Urban,2024-02-10 20:20:00,Saturday,Winter,Late Morning,Medium,4,3,92,17,22,66,105,Discharged,1
1,HOSP-3-20241128-0001,PAT-00002,HOSP-3,Northside Community Hospital,Rural,2024-11-28 02:07:00,Thursday,Fall,Evening,Medium,4,0,38,9,30,30,69,Discharged,3
2,HOSP-3-20240930-0002,PAT-00003,HOSP-3,Northside Community Hospital,Rural,2024-09-30 04:02:00,Monday,Fall,Evening,Low,5,1,38,38,40,125,203,Discharged,1
3,HOSP-2-20240227-0001,PAT-00004,HOSP-2,Riverside Medical Center,Urban,2024-02-27 00:31:00,Tuesday,Winter,Evening,High,4,5,94,8,16,64,88,Discharged,2
4,HOSP-1-20240306-0002,PAT-00005,HOSP-1,Springfield General Hospital,Urban,2024-03-06 16:52:00,Wednesday,Spring,Afternoon,Low,4,8,74,26,29,63,118,Discharged,1


# Emergency Room Wait Time Analysis

## Project Overview
This project analyses emergency room visit data from 20 hospitals across multiple regions,
covering 5000 patient visits. The goal is to identify key factors driving ER wait times,
uncover patterns in patient outcomes, and highlight hospitals that need operational intervention.

## Dataset
- **Source:** Kaggle — ER Wait Time Dataset
- **Size:** 5000 rows, 19 features
- **Domain:** Healthcare Operations

## Tools Used
- **Python & Pandas** — data preprocessing and star schema design
- **DuckDB** — analytical SQL database for querying

## Data Model
The dataset was broken down into a star schema with:
- `fact_er_visits` — one row per ER visit, containing all measurable metrics
- `dim_patient` — unique patients
- `dim_hospital` — hospital name and region
- `dim_time` — date, day of week, season, and time of day

This design separates descriptive attributes from measurable facts, making queries
more efficient and the data model easier to maintain.

## Exploratory Data Analysis
The goal of this section is to understand the basic composition of the dataset before
drawing any conclusions how visits are distributed, where the data is concentrated,
and whether any immediate patterns are visible.

### Q1 Distribution of Visits Across Urgency Levels
Before analysing wait times, it is important to understand how visits are distributed
across urgency levels. An imbalanced distribution could skew downstream analysis
for example, if 80% of visits are Low urgency, averages will naturally reflect that group.

In [40]:
# EDA Q1: Distribution of visits across Urgency Levels
# Goal: Understand the volume composition of the dataset by urgency level
con.execute("""
    SELECT
        "Urgency Level",
        COUNT("Visit ID") AS total_visits,
        ROUND(COUNT("Visit ID") * 100.0 / (SELECT COUNT(*) FROM fact_er_visits), 2)  AS percentage
    FROM fact_er_visits
    GROUP BY "Urgency Level"
    ORDER BY total_visits DESC
""").df()

,Urgency Level,total_visits,percentage
0,Medium,1291,25.82
1,High,1245,24.90
2,Critical,1242,24.84
3,Low,1222,24.44


### Q2 Wait Time Variation by Urgency Level vs Overall Average
A well-functioning ER should prioritise critical patients, meaning Critical urgency
visits should have shorter wait times than Low urgency ones. This query checks whether
urgency level is actually influencing how fast patients are seen, by comparing each
group's average against the overall average.

In [41]:
con.execute("""
    SELECT
        "Urgency Level",
        ROUND(AVG("Total Wait Time (min)"), 2) AS avg_wait,
        ROUND((SELECT AVG("Total Wait Time (min)") FROM fact_er_visits), 2) AS overall_avg,
        ROUND(AVG("Total Wait Time (min)") - (SELECT AVG("Total Wait Time (min)") FROM fact_er_visits), 2) AS diff_from_overall
    FROM fact_er_visits
    GROUP BY "Urgency Level"
    ORDER BY avg_wait DESC
""").df()

,Urgency Level,avg_wait,overall_avg,diff_from_overall
0,Low,173.54,81.92,91.62
1,Medium,93.70,81.92,11.78
2,High,43.19,81.92,-38.73
3,Critical,18.35,81.92,-63.57


### Q3 Hospital Visit Volume and Patient Satisfaction
High visit volume puts pressure on staff and resources. This query checks whether
the busiest hospitals are maintaining quality of care, or whether volume is coming
at the cost of patient satisfaction.

In [42]:
# EDA Q3: Which hospitals get the most visits and what is their avg satisfaction?
# Goal: Check whether high volume hospitals are maintaining quality of care
con.execute("""
    SELECT
        h."Hospital Name", h."Region",
        COUNT(f."Visit ID") AS total_visits,
        ROUND(AVG(f."Patient Satisfaction"), 2) AS avg_satisfaction
    FROM fact_er_visits f
    JOIN dim_hospital h
    ON f."Hospital ID" = h."Hospital ID"
    GROUP BY h."Hospital Name", h."Region"
    ORDER BY total_visits DESC
""").df()

,Hospital Name,Region,total_visits,avg_satisfaction
0,Riverside Medical Center,Urban,1023,2.76
1,Northside Community Hospital,Rural,999,2.81
2,St. Mary’s Regional Health,Rural,995,2.78
3,Springfield General Hospital,Urban,994,2.73
4,Summit Health Center,Urban,989,2.78


### Q4 Visit Distribution Across Season and Time of Day
Understanding when the ER is busiest is critical for staffing decisions. This query
breaks down visit volume by season and time of day simultaneously, revealing which
combinations drive peak demand.

In [43]:
# EDA Q4: Visit distribution across Season and Time of Day
# Goal: Identify peak demand periods — useful for staffing and resource planning
con.execute("""
    SELECT
        t."Season", t."Time of Day",
        COUNT(f."Visit ID") AS total_visits,
        ROUND(COUNT(f."Visit ID") * 100.0 / (SELECT COUNT(*) FROM fact_er_visits), 2) AS percentage
    FROM fact_er_visits f
    JOIN dim_time t
    ON f."Visit Date" = t."Visit Date"
    GROUP BY t."Season", t."Time of Day"
    ORDER BY t."Season", total_visits DESC
""").df()

,Season,Time of Day,total_visits,percentage
0,Fall,Evening,437,8.74
1,Fall,Afternoon,378,7.56
2,Fall,Late Morning,181,3.62
3,Fall,Night,132,2.64
4,Fall,Early Morning,110,2.20
5,Spring,Evening,414,8.28
6,Spring,Afternoon,362,7.24
7,Spring,Late Morning,197,3.94
8,Spring,Night,137,2.74
9,Spring,Early Morning,127,2.54


## Analysis
With the basic composition of the data understood, this section goes deeper
joining across tables, comparing subgroups, and using conditional aggregation
to derive insights.

### Q5 Regional Wait Time Breakdown
Not all regions perform equally. This query identifies which regions have the longest
average wait times, and breaks down whether the delay is happening at registration,
triage, or the time to see a medical professional. Knowing the bottleneck is the
first step toward fixing it.

In [44]:
# ── Q5: Which regions have the longest wait time and what is driving it? ──
# Goal: Pinpoint whether delays come from registration, triage, or medical staff
con.execute("""
    SELECT
        h."Region",
        ROUND(AVG(f."Total Wait Time (min)"), 2) AS avg_total_wait,
        ROUND(AVG(f."Time to Registration (min)"), 2) AS avg_registration,
        ROUND(AVG(f."Time to Triage (min)"), 2) AS avg_triage,
        ROUND(AVG(f."Time to Medical Professional (min)"), 2) AS avg_to_medical
    FROM fact_er_visits f
    JOIN dim_hospital h
    ON f."Hospital ID" = h."Hospital ID"
    GROUP BY h."Region"
    ORDER BY avg_total_wait DESC
""").df()

,Region,avg_total_wait,avg_registration,avg_triage,avg_to_medical
0,Urban,81.98,11.74,24.71,45.53
1,Rural,81.82,11.65,25.01,45.17


### Q6 Impact of Staffing Levels on Wait Time
Intuitively, more nurses and specialists should mean shorter wait times. This query
tests that assumption by bucketing Nurse-to-Patient Ratio and Specialist Availability
into Low, Medium, and High groups, then comparing average wait times across combinations.
If staffing level has no effect, it suggests the bottleneck lies elsewhere
in processes or infrastructure rather than headcount.

In [46]:
# ── Q6: Does staffing level impact wait time? ─────────────────────────────
# Step 1: Understand the range of values before bucketing
con.execute("""
    SELECT
        MIN("Nurse-to-Patient Ratio") AS min_nurse_ratio,
        MAX("Nurse-to-Patient Ratio") AS max_nurse_ratio,
        MIN("Specialist Availability") AS min_specialist,
        MAX("Specialist Availability") AS max_specialist
    FROM fact_er_visits
""").df()

,min_nurse_ratio,max_nurse_ratio,min_specialist,max_specialist
0,1,5,0,10


In [47]:
# Step 2: Bucket staffing levels and compare average wait times
con.execute("""
    SELECT
        CASE
            WHEN "Nurse-to-Patient Ratio" <= 3 THEN 'Low'
            WHEN "Nurse-to-Patient Ratio" <= 6 THEN 'Medium'
            ELSE 'High'
        END AS nurse_ratio_bucket,
        CASE
            WHEN "Specialist Availability" <= 3 THEN 'Low'
            WHEN "Specialist Availability" <= 6 THEN 'Medium'
            ELSE 'High'
        END AS specialist_bucket, COUNT("Visit ID") AS total_visits,
        ROUND(AVG("Total Wait Time (min)"), 2) AS avg_wait
    FROM fact_er_visits
    GROUP BY nurse_ratio_bucket, specialist_bucket
    ORDER BY avg_wait DESC
""").df()

,nurse_ratio_bucket,specialist_bucket,total_visits,avg_wait
0,Medium,Low,1426,125.99
1,Medium,High,568,123.23
2,Medium,Medium,441,119.85
3,Low,High,589,42.63
4,Low,Low,1504,41.49
5,Low,Medium,472,41.44


### Q7 Patient Outcomes by Urgency Level
This query measures the percentage of each patient outcome within each urgency level.
Ideally, Critical urgency patients should have the best outcomes since they receive
priority care. If Low urgency patients are achieving better outcomes than Critical ones,
it signals a serious triage problem that goes beyond wait times.

In [49]:
# ── Q7: Patient outcome breakdown by Urgency Level
# Goal: Check whether critical patients achieve better outcomes than low urgency ones
con.execute("""
    SELECT
        "Urgency Level", "Patient Outcome",
        COUNT("Visit ID") AS total_visits,
        ROUND(
            COUNT("Visit ID") * 100.0 / SUM(COUNT("Visit ID")) OVER (PARTITION BY "Urgency Level")
        , 2) AS pct_within_urgency
    FROM fact_er_visits
    GROUP BY "Urgency Level", "Patient Outcome"
    ORDER BY "Urgency Level", total_visits DESC
""").df()

,Urgency Level,Patient Outcome,total_visits,pct_within_urgency
0,Critical,Admitted,626,50.40
1,Critical,Discharged,616,49.60
2,High,Admitted,632,50.76
3,High,Discharged,613,49.24
4,Low,Discharged,918,75.12
5,Low,Left Without Being Seen,198,16.20
6,Low,Admitted,106,8.67
7,Medium,Discharged,732,56.70
8,Medium,Admitted,504,39.04
9,Medium,Left Without Being Seen,55,4.26


### Q8 Hospital Wait Time Ranking Within Each Region
Ranking hospitals within their own region gives a fairer comparison than a global ranking, since regional factors like population density and funding affect baseline wait times. This query uses DENSE_RANK() partitioned by region to identify which hospital is the worst performer in each area.


In [51]:
# Q8: How does each hospital's wait time rank within its region?
# Concepts: CTE + DENSE_RANK partitioned by region
con.execute("""
    WITH hospital_avg AS (
        SELECT
            h."Hospital Name", h."Region",
            ROUND(AVG(f."Total Wait Time (min)"), 2) AS avg_wait
        FROM fact_er_visits f
        JOIN dim_hospital h
        ON f."Hospital ID" = h."Hospital ID"
        GROUP BY h."Hospital Name", h."Region"
    )
    SELECT
        "Hospital Name", "Region", avg_wait,
        DENSE_RANK() OVER (PARTITION BY "Region" ORDER BY avg_wait DESC) AS rank_within_region
    FROM hospital_avg
    ORDER BY "Region", rank_within_region
""").df()

,Hospital Name,Region,avg_wait,rank_within_region
0,St. Mary’s Regional Health,Rural,81.89,1
1,Northside Community Hospital,Rural,81.76,2
2,Springfield General Hospital,Urban,82.70,1
3,Riverside Medical Center,Urban,81.81,2
4,Summit Health Center,Urban,81.43,3


### Q9 Month-over-Month Trend in Average Wait Time
A single average hides whether things are getting better or worse over time.
This query uses the LAG window function to compute the month-over-month change
in average wait time, revealing whether ER performance is improving,
deteriorating, or staying flat across the observation period.

In [52]:
# Q9: What is the month-over-month trend in average wait time?
# Concepts: CTE + STRFTIME + LAG window function
con.execute("""
    WITH monthly_avg AS (
        SELECT
            STRFTIME(CAST(t."Visit Date" AS TIMESTAMP), '%Y-%m') AS visit_month,
            ROUND(AVG(f."Total Wait Time (min)"), 2) AS avg_wait
        FROM fact_er_visits f
        JOIN dim_time t
        ON f."Visit Date" = t."Visit Date"
        GROUP BY visit_month
    )
    SELECT
        visit_month, avg_wait,
        LAG(avg_wait) OVER (ORDER BY visit_month) AS prev_month_avg,
        ROUND(avg_wait - LAG(avg_wait) OVER (ORDER BY visit_month), 2) AS mom_change
    FROM monthly_avg
    ORDER BY visit_month
""").df()

,visit_month,avg_wait,prev_month_avg,mom_change
0,2024-01,97.04,NaN,NaN
1,2024-02,96.37,97.04,-0.67
2,2024-03,68.85,96.37,-27.52
3,2024-04,74.97,68.85,6.12
4,2024-05,72.54,74.97,-2.43
5,2024-06,85.44,72.54,12.90
6,2024-07,93.58,85.44,8.14
7,2024-08,90.04,93.58,-3.54
8,2024-09,64.85,90.04,-25.19
9,2024-10,66.11,64.85,1.26
